# 05d_Hierarchical_nivel2


In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering' / 'nivel2'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'features_tier2.parquet'
TIER_LEVEL = '2'

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object','category']).columns.tolist():
        if out[cat].nunique(dropna=True) > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess(df, nan_threshold_zero=0.30):
    df = df.drop(columns=['user_id']).copy()
    numeric = df.select_dtypes(include=['number']).columns.tolist()
    categorical = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    df = reduce_high_card(df, 10)
    pp = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical),
    ])
    X = pp.fit_transform(df)
    return X, pp, pp.get_feature_names_out()

print(f'TIER {TIER_LEVEL}: cargando {MASTER_PATH.name}')
master = pd.read_parquet(MASTER_PATH)
user_ids = master['user_id'].values
print(f'  shape: {master.shape}')
X, preproc, names = preprocess(master)
joblib.dump(preproc, DATA_MODELS / f'preprocessor_tier{TIER_LEVEL}.joblib')
print(f'  X post-preproc: {X.shape}')
N = len(X)

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
REPORT = INFORMES_BASE / '05d_hierarchical'
REPORT.mkdir(parents=True, exist_ok=True)
K_RANGE = list(range(2, 11))
SAMPLE_H = min(20_000, len(X))
SAMPLE_SIL = 20_000
results = []


TIER 2: cargando features_tier2.parquet
  shape: (21599, 17)
  X post-preproc: (21599, 16)


In [2]:
rng = np.random.RandomState(42)
if len(X) > SAMPLE_H:
    sample_idx = rng.choice(len(X), SAMPLE_H, replace=False)
    X_sample = X[sample_idx]
else:
    sample_idx = np.arange(len(X))
    X_sample = X

print(f'  Linkage sobre {len(X_sample):,}...')
t = time.time()
Z = linkage(X_sample, method='ward')
print(f'  linkage en {time.time()-t:.1f}s')

fig, ax = plt.subplots(figsize=(12,4))
dendrogram(Z, truncate_mode='level', p=5, ax=ax, no_labels=True)
ax.set_title(f'Dendrograma nivel{TIER_LEVEL} (sample {len(X_sample):,})')
plt.tight_layout(); plt.savefig(REPORT / 'dendrogram.png', dpi=120, bbox_inches='tight'); plt.close()

sil_idx = rng.choice(len(X), min(SAMPLE_SIL, len(X)), replace=False)
X_sil = X[sil_idx]
for k in K_RANGE:
    t0 = time.time()
    labels_sample = fcluster(Z, t=k, criterion='maxclust')
    centroids = np.array([X_sample[labels_sample == i].mean(axis=0) for i in range(1, k+1)])
    km = KMeans(n_clusters=k, init=centroids, n_init=1, random_state=42)
    labels_all = km.fit_predict(X)
    labels_sub = km.predict(X_sil)
    sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub))>1 else float('nan')
    db = float(davies_bouldin_score(X, labels_all))
    unique, counts = np.unique(labels_all, return_counts=True)
    max_pct = float(counts.max()/len(labels_all)); min_pct = float(counts.min()/len(labels_all))
    results.append({'algorithm':'Hierarchical','tier':TIER_LEVEL,'k':k,'silhouette':sil,'davies_bouldin':db,'n_clusters_actual':int(len(set(labels_all))),'max_cluster_pct':max_pct,'min_cluster_pct':min_pct,'elapsed_s':time.time()-t0})
    print(f'  K={k}: sil={sil:.4f} db={db:.4f} max%={max_pct:.1%} min%={min_pct:.1%}')
    joblib.dump({'model':km,'Z':Z,'labels':labels_all,'user_ids':user_ids,'sample_idx':sample_idx}, DATA_MODELS / f'nivel{TIER_LEVEL}_hierarchical_k{k}.joblib')

pd.DataFrame(results).to_csv(REPORT / 'metrics.csv', index=False)
print('OK metrics.csv')


  Linkage sobre 20,000...


  linkage en 5.9s


  K=2: sil=0.9804 db=0.6056 max%=99.7% min%=0.3%


  K=3: sil=0.9780 db=0.4049 max%=99.6% min%=0.0%


  K=4: sil=0.9668 db=0.5651 max%=99.4% min%=0.0%


...

  K=6: sil=0.9292 db=0.6226 max%=98.4% min%=0.0%


  K=7: sil=0.9123 db=0.6635 max%=98.1% min%=0.0%


  K=8: sil=0.9124 db=0.5932 max%=98.1% min%=0.0%


  K=9: sil=0.9098 db=0.6124 max%=98.0% min%=0.0%


  K=10: sil=0.8746 db=0.6954 max%=97.2% min%=0.0%
OK metrics.csv
